# ChemBreak Colab Generator V4

V4 performs a controlled four-family generator and judge comparison.

- Family A generates and is judged by A, B, C, D.
- Family B generates and is judged by A, B, C, D.
- Family C generates and is judged by A, B, C, D.
- Family D generates and is judged by A, B, C, D.

Start with `run_config_smoke.json`.

The smoke run uses one matrix row, two candidate positions, and all four families. This gives 8 generated candidates and 32 blind judgments.

After that succeeds, switch to `run_config.json`.

The full V4 pilot uses 18 matrix rows × 5 candidates × 4 generators = 360 generated candidates, followed by 1,440 blind judgments.


In [ ]:
# 1. Confirm GPU runtime.

import torch
import platform

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected. In Colab choose "
        "Runtime > Change runtime type > GPU."
    )

print("GPU:", torch.cuda.get_device_name(0))
print(
    "GPU memory (GB):",
    round(
        torch.cuda.get_device_properties(0).total_memory
        / 1024**3,
        2,
    ),
)


In [ ]:
# 2. Clone the ChemBreak GitHub repository.

from pathlib import Path
import subprocess

REPO_URL = "https://github.com/Jollychuks/ChemBreak.git"
BRANCH = "main"
PROJECT_SUBDIR = "ChemBreak_Colab_Generator_v4"

CLONE_DIR = Path("/content/chembreak_repo")

if not CLONE_DIR.exists():
    subprocess.run(
        [
            "git",
            "clone",
            "--branch",
            BRANCH,
            REPO_URL,
            str(CLONE_DIR),
        ],
        check=True,
    )
else:
    print("Repository already cloned:", CLONE_DIR)

PROJECT_DIR = (
    CLONE_DIR / PROJECT_SUBDIR
).resolve()

if not PROJECT_DIR.exists():
    raise FileNotFoundError(
        f"V4 folder not found: {PROJECT_DIR}\n"
        "Upload the complete ChemBreak_Colab_Generator_v4 "
        "folder to GitHub first."
    )

print("PROJECT_DIR:", PROJECT_DIR)

print("\nFiles:")
for p in sorted(PROJECT_DIR.iterdir()):
    print(" -", p.name)


## Install the current model runtimes

Qwen3.5, Ministral 3, and Gemma 4 require recent Hugging Face Transformers support. V4 pins Transformers 5.15.0 so separate Colab sessions use the same runtime version.

The four models are never loaded simultaneously. V4 loads one family, completes its stage, unloads it, clears GPU memory, and then loads the next family.


In [ ]:
# 3. Install V4 dependencies.

import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-U",
        "-r",
        str(PROJECT_DIR / "requirements_colab.txt"),
    ],
    check=True,
)

print("Dependencies installed.")
print(
    "If a later import reports a stale Transformers version, "
    "restart the Colab runtime once, then rerun from Cell 1."
)


## Optional Hugging Face authentication

Most model files can be downloaded directly. Some repositories may require that you accept model terms on Hugging Face and authenticate.

Do not put a Hugging Face token in the notebook or GitHub repository.

If needed, set `USE_HF_TOKEN = True`. The token is entered with hidden input and exists only in the runtime.


In [ ]:
# 4. OPTIONAL Hugging Face token.

USE_HF_TOKEN = False

HF_TOKEN = None

if USE_HF_TOKEN:
    from getpass import getpass
    from huggingface_hub import login

    HF_TOKEN = getpass(
        "Hugging Face token (input hidden): "
    )

    login(
        token=HF_TOKEN,
        add_to_git_credential=False,
    )

    print("Hugging Face authentication active for this runtime.")
else:
    print("Using anonymous Hugging Face access.")


## Choose experiment configuration

Keep `CONFIG_FILE = "run_config_smoke.json"` until all four model families successfully generate and judge.

Then change the line to:

`CONFIG_FILE = "run_config.json"`

The smoke and full pilot use separate output folders, so the files cannot accidentally mix.


In [ ]:
# 5. Import V4 and load the selected configuration.

import sys
import json
import importlib

sys.path.insert(
    0,
    str(PROJECT_DIR),
)

import chembreak_common
import scenario_controller
import model_runtime
import experiment_manifest
import generate_multimodel
import judge_multimodel
import aggregate_comparison
import human_review
import freeze_environment

for module in [
    chembreak_common,
    scenario_controller,
    model_runtime,
    experiment_manifest,
    generate_multimodel,
    judge_multimodel,
    aggregate_comparison,
    human_review,
    freeze_environment,
]:
    importlib.reload(module)

CONFIG_FILE = "run_config_smoke.json"
# After the smoke run succeeds, change the line above to:
# CONFIG_FILE = "run_config.json"

config = chembreak_common.load_json(
    PROJECT_DIR / CONFIG_FILE
)

registry = chembreak_common.load_json(
    PROJECT_DIR / config["model_registry_file"]
)

taxonomy = chembreak_common.load_json(
    PROJECT_DIR / config["taxonomy_file"]
)

matrix = chembreak_common.load_matrix(
    PROJECT_DIR / config["matrix_file"]
)

print("Experiment:", config["experiment_id"])
print("Package:", config["package_version"])
print("Generator prompt:", config["generator_prompt_version"])
print("Judge prompt:", config["judge_prompt_version"])
print("Scenario plan:", config["scenario_plan_version"])
print("Candidates per row:", config["n_per_row"])

print("\nModel families:")
for family_id, family in registry["families"].items():
    print(
        f"{family_id}: {family['family_name']} "
        f"=> {family['model_id']}"
    )


In [ ]:
# 6. Inspect the controlled matrix selection and expected run size.

selected = chembreak_common.select_rows(
    matrix,
    config,
)

generator_count = len(
    config["generation_families"]
)

judge_count = len(
    config["judge_families"]
)

candidate_count = (
    len(selected)
    * int(config["n_per_row"])
    * generator_count
)

judgment_count = (
    candidate_count
    * judge_count
)

print("Selected matrix rows:", len(selected))
print("Expected generated candidates:", candidate_count)
print("Expected blind judgments:", judgment_count)

display(
    selected[
        [
            "MATRIX_ID",
            "HC_ID",
            "HC_CATEGORY",
            "HD_ID",
            "HAZARD_DOMAIN",
            "OT_ID",
            "OUTPUT_TYPE",
            "ALLOWED_SCENARIOS",
        ]
    ]
)


## Shared scenario plan

Python creates the scenario plan once before any family generates.

All four generators therefore receive the same scenario assignment for the same matrix row and candidate index.

The plan is persisted and its configuration signature is checked on resume.


In [ ]:
# 7. Materialize and inspect the shared scenario plan.

scenario_plan = scenario_controller.ensure_scenario_plan(
    PROJECT_DIR,
    config,
    matrix,
)

print("Scenario-plan rows:", len(scenario_plan))
display(scenario_plan.head(30))


In [ ]:
# 8. Create the experiment provenance manifest.

manifest = experiment_manifest.create_experiment_manifest(
    PROJECT_DIR,
    CLONE_DIR,
    config,
    config_filename=CONFIG_FILE,
)

print(json.dumps(manifest, indent=2))


## Optional automatic GitHub checkpoints

Local CSV checkpointing is always immediate.

GitHub checkpointing is optional. If enabled in the config, V4 can push after a configured number of successful candidates or judgments.

The GitHub token is entered only at runtime.


In [ ]:
# 9. Optional GitHub checkpoint token.

AUTO_GITHUB_CHECKPOINT = bool(
    config["github_checkpoint"].get(
        "enabled",
        False,
    )
)

GITHUB_TOKEN = None

if AUTO_GITHUB_CHECKPOINT:
    from getpass import getpass

    GITHUB_TOKEN = getpass(
        "GitHub token (input hidden): "
    )

print(
    "Automatic GitHub checkpoints:",
    AUTO_GITHUB_CHECKPOINT,
)


## Generation phase

The defaults run all four generators sequentially.

For a long full-pilot run, you can execute one family per Colab session by changing `GENERATOR_FAMILIES_TO_RUN`, for example `["A"]`, then later `["B"]`. Resume-safe candidate IDs prevent duplication.


In [ ]:
# 10. Select generator families for this session.

GENERATOR_FAMILIES_TO_RUN = list(
    config["generation_families"]
)

print(
    "Generator families for this session:",
    GENERATOR_FAMILIES_TO_RUN,
)


In [ ]:
# 11. Run the selected generator families.

candidate_path = generate_multimodel.run_all_generators(
    PROJECT_DIR,
    CLONE_DIR,
    config,
    registry,
    taxonomy,
    matrix,
    family_ids=GENERATOR_FAMILIES_TO_RUN,
    github_token=GITHUB_TOKEN,
    hf_token=HF_TOKEN,
)

print("\nCandidate dataset:", candidate_path)


In [ ]:
# 12. Review generated candidate coverage.

import pandas as pd

candidates = pd.read_csv(
    candidate_path
).fillna("")

current_candidates = candidates[
    candidates["experiment_id"].astype(str)
    == str(config["experiment_id"])
].copy()

print(
    "Generated candidates for current experiment:",
    len(current_candidates),
)

print("\nBy generator:")
display(
    current_candidates.groupby(
        [
            "generator_family_id",
            "generator_family_name",
        ]
    )["candidate_id"]
    .count()
    .to_frame("candidates")
)

print("\nRich candidate preview:")
display(
    current_candidates[
        [
            "candidate_id",
            "generator_family_id",
            "matrix_id",
            "hc_id",
            "hc_category",
            "hd_id",
            "hazard_domain",
            "ot_id",
            "output_type",
            "allowed_scenarios",
            "selected_scenarios",
            "benchmark_prompt",
            "main_goal",
            "chemical_entity",
            "distinctive_dimension",
            "generation_attempts",
            "generator_prompt_version",
        ]
    ].tail(40)
)


## Blind judging phase

Every candidate is judged by all four families.

The judge prompt does not receive generator family, generator model, candidate ID, main_goal, chemical_entity, or distinctive_dimension.

It receives only the controlled matrix definitions, scenario assignment, and benchmark_prompt.

Each judge returns eight independent 1 to 5 scores plus a short reason.

Python, not the model, converts the scores to ACCEPT, REVISE, or REJECT.


In [ ]:
# 13. Select judge families for this session.

JUDGE_FAMILIES_TO_RUN = list(
    config["judge_families"]
)

print(
    "Judge families for this session:",
    JUDGE_FAMILIES_TO_RUN,
)


In [ ]:
# 14. Run the selected blind judge families.

judgment_path = judge_multimodel.run_all_judges(
    PROJECT_DIR,
    CLONE_DIR,
    config,
    registry,
    taxonomy,
    family_ids=JUDGE_FAMILIES_TO_RUN,
    github_token=GITHUB_TOKEN,
    hf_token=HF_TOKEN,
)

print("\nJudgment dataset:", judgment_path)


In [ ]:
# 15. Review judgment coverage and score distributions.

judgments = pd.read_csv(
    judgment_path
).fillna("")

current_judgments = judgments[
    judgments["experiment_id"].astype(str)
    == str(config["experiment_id"])
].copy()

print(
    "Total judgments for current experiment:",
    len(current_judgments),
)

print("\nDecision counts by judge:")
display(
    current_judgments.groupby(
        [
            "judge_family_id",
            "validator_decision",
        ]
    )["judgment_id"]
    .count()
    .unstack(fill_value=0)
)

print("\nAverage scores by judge:")
display(
    current_judgments.groupby(
        "judge_family_id"
    )[
        judge_multimodel.SCORE_COLUMNS
    ].mean()
)

print("\nRecent judgments:")
display(
    current_judgments[
        [
            "judgment_id",
            "candidate_id",
            "generator_family_id",
            "judge_family_id",
            "harmful_intent_score",
            "chemistry_dependency_score",
            "hc_fit_score",
            "hd_fit_score",
            "ot_fit_score",
            "chemistry_plausibility_score",
            "scenario_consistency_score",
            "jailbreak_readiness_score",
            "overall_quality_score",
            "validator_decision",
            "judge_reason",
        ]
    ].tail(40)
)


## Four-family comparison

V4 reports both all-judge and cross-family measurements.

All-judge results use the same four-judge panel for every generator.

Cross-family results exclude the generator's own family judge and are useful for checking whether self-family judging changes the conclusion.

Do not select the final judges only from model-model agreement.


In [ ]:
# 16. Aggregate the generator and judge comparison.

result_files = aggregate_comparison.aggregate_results(
    PROJECT_DIR,
    config,
    matrix=matrix,
)

for name, path in result_files.items():
    print(name, "=>", path)

generator_summary = pd.read_csv(
    result_files["generator_summary"]
)

judge_summary = pd.read_csv(
    result_files["judge_summary"]
)

print("\nGenerator summary:")
display(generator_summary)

print("\nJudge behavior summary:")
display(judge_summary)

print("\nGenerator by judge matrix:")
display(
    pd.read_csv(
        result_files["generator_by_judge_matrix"]
    )
)

print("\nPairwise judge agreement:")
display(
    pd.read_csv(
        result_files["judge_pairwise_agreement"]
    )
)


## Blinded human calibration

Model agreement is not ground truth.

The full pilot creates a blinded sample with one task per HC category per generator family, normally 36 tasks.

Generator identity is stored in a separate key file. Do not inspect that key before labeling.


In [ ]:
# 17. Create blinded human-review files.

blinded_path, key_path = (
    human_review.create_blinded_human_sample(
        PROJECT_DIR,
        config,
        taxonomy,
    )
)

print("Blinded review file:", blinded_path)
print("Hidden key file:", key_path)
print(
    "Use HUMAN_REVIEW_GUIDE.md while labeling. "
    "Do not open the key until labels are complete."
)


After you fill the eight `human_*_score` columns and `human_decision` in the blinded CSV, run the next cell.

For the strongest methodology, use two independent human reviewers if feasible and adjudicate disagreements before treating the labels as reference judgments.


In [ ]:
# 18. OPTIONAL after manual human labeling is complete.

RUN_HUMAN_EVALUATION = False

if RUN_HUMAN_EVALUATION:
    human_result_files = (
        human_review.evaluate_judges_against_human(
            PROJECT_DIR,
            config,
        )
    )

    print("\nJudges compared with human reference labels:")
    display(
        pd.read_csv(
            human_result_files["judge_vs_human"]
        )
    )

    print("\nGenerators compared directly with human reference labels:")
    display(
        pd.read_csv(
            human_result_files["generator_vs_human"]
        )
    )


In [ ]:
# 19. Freeze the successful experiment environment.

freeze_path = freeze_environment.freeze_environment(
    chembreak_common.resolve_path(
        PROJECT_DIR,
        config["output_dir"],
    )
    / "environment_freeze.txt"
)

print("Environment freeze:", freeze_path)


## Moving from smoke to full V4 pilot

When the smoke run works for A, B, C, and D:

1. Change `CONFIG_FILE` to `run_config.json`.
2. Rerun from the configuration cell.
3. The full pilot writes to `outputs/`, while the smoke run remains in `outputs_smoke/`.
4. If the full run is too long for one Colab session, run one generator family or one judge family per session.
5. Resume is supported.
6. If automatic GitHub checkpointing is enabled, the output files are periodically pushed to GitHub.

Do not move to thousands of final ChemBreak tasks until the four-family generator comparison and human-calibrated judge comparison are reviewed.
